# Part D — Policy RAG Assistant and Refund Agent

This executable, text-only notebook completes all nine Part D tasks (28 marks). It uses the five supplied policy documents, the Part A SQLite database, the 14-question policy evaluation and all seven refund scenarios. Every model response comes from the unchanged offline `mock_llm.py`; no API key or network service is needed.

Run cells from top to bottom. The notebook locates the repository whether launched from its root or `part_d/`. Run Part A first to create `data/instamart.db`. Executions regenerate Part D reports and named demonstration checkpoints; existing unrelated checkpoint files are preserved. The final cells verify all acceptance criteria and explain the measured ReAct tradeoff.

In [1]:
from pathlib import Path
import json
import sys
from collections import Counter
import pandas as pd

ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / 'mock_llm.py').is_file() and (candidate / 'part_d').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the project root or part_d folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert (ROOT / 'data/instamart.db').is_file(), 'Run Part A before this notebook.'

import mock_llm
from part_d.rag import chunk_policy_documents, PolicyRetriever, evaluate_policy_rag
from part_d.tools import TOOLS_SPEC, TOOL_MAP, get_order, search_policy, calculate_refund, issue_refund
from part_d.agent import SimulatedCrash, run_agent, resume_agent, load_checkpoint
from part_d.run_part_d import build_report, refund_from_result

pd.set_option('display.max_colwidth', 80)
print('Repository detected:', ROOT.name)
print('Offline model:', Path(mock_llm.__file__).name)
print('Part A database available:', (ROOT / 'data/instamart.db').is_file())

Repository detected: Capstone _Project
Offline model: mock_llm.py
Part A database available: True


## Task 1 — Chunk the documents (2 marks)

A chunk contains at most 40 whitespace-separated words. With 10-word overlap, starts advance by 30 words: 0, 30, 60, and so on. We stop immediately after the first chunk reaching the final word; `end_word` below is exclusive, so the first full chunk covers words 0–39. Each chunk records the policy filename and its exact text.

In [2]:
policy_files = sorted((ROOT / 'data/policies').glob('*.txt'))
assert len(policy_files) == 5
chunks_10 = chunk_policy_documents(chunk_size=40, overlap=10)
counts_10 = dict(Counter(chunk['source'] for chunk in chunks_10))
expected_counts = {'refund_policy.txt': 4, 'delivery_policy.txt': 3, 'escalation_policy.txt': 3,
                   'payment_policy.txt': 3, 'privacy_policy.txt': 2}
assert counts_10 == expected_counts and len(chunks_10) == 15
policy_inventory = pd.DataFrame([
    {'source': path.name, 'document_words': len(path.read_text().split()),
     'chunks': counts_10[path.name]} for path in policy_files
])
print(policy_inventory.to_string(index=False))
print('Total chunks:', len(chunks_10))

               source  document_words  chunks
  delivery_policy.txt             100       3
escalation_policy.txt              78       3
   payment_policy.txt              86       3
   privacy_policy.txt              68       2
    refund_policy.txt             119       4
Total chunks: 15


In [3]:
chunk_table = pd.DataFrame([
    {'source': chunk['source'], 'start_word': chunk['start_word'],
     'end_word_exclusive': chunk['end_word'], 'chunk_words': len(chunk['text'].split())}
    for chunk in chunks_10
])
for source in counts_10:
    document_chunks = [chunk for chunk in chunks_10 if chunk['source'] == source]
    assert [chunk['start_word'] for chunk in document_chunks] == list(range(0, 30 * len(document_chunks), 30))
    assert all(0 < len(chunk['text'].split()) <= 40 for chunk in document_chunks)
print(chunk_table.to_string(index=False))

               source  start_word  end_word_exclusive  chunk_words
  delivery_policy.txt           0                  40           40
  delivery_policy.txt          30                  70           40
  delivery_policy.txt          60                 100           40
escalation_policy.txt           0                  40           40
escalation_policy.txt          30                  70           40
escalation_policy.txt          60                  78           18
   payment_policy.txt           0                  40           40
   payment_policy.txt          30                  70           40
   payment_policy.txt          60                  86           26
   privacy_policy.txt           0                  40           40
   privacy_policy.txt          30                  68           38
    refund_policy.txt           0                  40           40
    refund_policy.txt          30                  70           40
    refund_policy.txt          60                 100         

## Task 2 — Vectorise and retrieve (3 marks)

TF measures how frequently a term occurs in a chunk, while IDF gives more weight to terms occurring in fewer chunks across the corpus. TF-IDF relies on shared words, so semantically equivalent phrases such as “money back” and “refund” can have low similarity when their vocabulary differs.

We fit scikit-learn's default `TfidfVectorizer()` once on the chunk text. Queries use that fitted vocabulary, and `cosine_similarity` ranks chunks; `retrieve(query, k=3)` returns source, score and text dictionaries.

In [4]:
retriever_10 = PolicyRetriever(chunks_10)
query = 'refund policy late delivery'
top_three = retriever_10.retrieve(query, k=3)
assert len(top_three) == 3
assert [result['source'] for result in top_three] == ['refund_policy.txt'] * 3
assert all(set(result) == {'source', 'score', 'text'} for result in top_three)
print('TF-IDF matrix (chunks, vocabulary terms):', retriever_10.tfidf_matrix.shape)
print(pd.DataFrame([{'rank': rank, 'source': item['source'], 'cosine_score': round(item['score'], 6)}
                    for rank, item in enumerate(top_three, 1)]).to_string(index=False))
for rank, item in enumerate(top_three, 1):
    print(f"\nRetrieved chunk {rank} [{item['source']}]: {item['text']}")

TF-IDF matrix (chunks, vocabulary terms): (15, 200)
 rank            source  cosine_score
    1 refund_policy.txt      0.271928
    2 refund_policy.txt      0.251374
    3 refund_policy.txt      0.245406

Retrieved chunk 1 [refund_policy.txt]: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are

Retrieved chunk 2 [refund_policy.txt]: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

Retrieved chunk 3 [refund_policy.txt]: late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR 1000 must be approved by a human support lead before money is returned. Approved refunds are credited

## Task 3 — Augment and answer (3 marks)

`answer(question)` sends exactly two messages to `mock_llm.chat`: a system instruction containing `POLICY ASSISTANT` and a user message in the required `CONTEXT:` / `[source]` / `QUESTION:` format. Answers must use the supplied context and begin “I could not find” when the information is absent. Returned source names come from retrieval rather than being invented by the model.

In [5]:
example_question = 'What delivery time makes a completed order eligible for a full refund?'
messages, prompt_sources = retriever_10.build_messages(example_question)
expected_context = '\n'.join(f"[{item['source']}]\n{item['text']}" for item in retriever_10.retrieve(example_question))
assert len(messages) == 2 and 'POLICY ASSISTANT' in messages[0]['content']
assert messages[1]['content'] == f'CONTEXT:\n{expected_context}\nQUESTION:\n{example_question}'
print('SYSTEM:', messages[0]['content'])
print('USER:\n' + messages[1]['content'])
answer_text, answer_sources, answer_tokens = retriever_10.answer(example_question)
assert 'more than 45 minutes' in answer_text
assert answer_sources == prompt_sources
print('\nAnswer:', answer_text)
print('Sources:', answer_sources)
print('Estimated tokens used:', answer_tokens)

SYSTEM: POLICY ASSISTANT. Answer only from the supplied context. If the information is missing, say I could not find that information in the policy documents.
USER:
CONTEXT:
[refund_policy.txt]
order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are
[refund_policy.txt]
Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer
[privacy_policy.txt]
Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account deletion and the request is completed within 30 days. Payment card numbers are never stored by the support system.
QUESTION:
What delivery time makes a completed order eligible for a full refund?

An

## Task 4 — Evaluate retrieval and grounding (3 marks)

For each of 12 answerable questions, measure whether the expected document occurs among the three retrieved source names and whether the answer contains the case-sensitive expected fact. For the two uncovered questions, measure the fallback prefix. The whole evaluation is then repeated with zero overlap; document retrieval alone is insufficient evidence that an answer is grounded correctly.

In [6]:
evaluation_10 = evaluate_policy_rag(retriever_10)
chunks_0 = chunk_policy_documents(chunk_size=40, overlap=0)
retriever_0 = PolicyRetriever(chunks_0)
evaluation_0 = evaluate_policy_rag(retriever_0)
comparison = pd.DataFrame([
    {'overlap_words': overlap, 'chunks': number,
     'expected_doc_in_top3': f"{evaluation['doc_hits']}/12",
     'expected_fact_in_answer': f"{evaluation['fact_hits']}/12",
     'uncovered_fallback': f"{evaluation['fallback_hits']}/2"}
    for overlap, number, evaluation in [(10, len(chunks_10), evaluation_10), (0, len(chunks_0), evaluation_0)]
])
assert (evaluation_10['doc_hits'], evaluation_10['fact_hits'], evaluation_10['fallback_hits']) == (12, 12, 2)
assert evaluation_0['fact_hits'] < 12
print(comparison.to_string(index=False))

 overlap_words  chunks expected_doc_in_top3 expected_fact_in_answer uncovered_fallback
            10      15                12/12                   12/12                2/2
             0      13                12/12                   10/12                2/2


In [7]:
full_question_results = pd.DataFrame([
    {'question_id': detail['question_id'], 'expected_doc': detail['expected_doc'],
     'doc_hit': detail['doc_hit'], 'fact_hit': detail['fact_hit'],
     'fallback_hit': detail['fallback_hit'], 'expected_fact': detail['expected_fact']}
    for detail in evaluation_10['details']
])
print(full_question_results.to_string(index=False))
for detail in evaluation_10['details']:
    print(f"\n{detail['question_id']}: {detail['question']}\nAnswer: {detail['answer']}")

question_id          expected_doc  doc_hit  fact_hit  fallback_hit          expected_fact
         Q1     refund_policy.txt     True      True         False   more than 45 minutes
         Q2     refund_policy.txt     True      True         False     human support lead
         Q3     refund_policy.txt     True      True         False          within 7 days
         Q4   delivery_policy.txt     True      True         False                 INR 25
         Q5   delivery_policy.txt     True      True         False        INR 199 or more
         Q6   delivery_policy.txt     True      True         False      one time password
         Q7    payment_policy.txt     True      True         False               INR 2500
         Q8    payment_policy.txt     True      True         False               48 hours
         Q9    payment_policy.txt     True      True         False never refunded as cash
        Q10    privacy_policy.txt     True      True         False               180 days
        Q1

In [8]:
zero_overlap_failures = [detail for detail in evaluation_0['details']
                         if detail['expected_doc'] != 'NONE' and not detail['fact_hit']]
assert {detail['question_id'] for detail in zero_overlap_failures} == {'Q1', 'Q8'}
for detail in zero_overlap_failures:
    print(f"{detail['question_id']}: {detail['question']}")
    print('Expected fact:', detail['expected_fact'])
    print('Retrieved sources:', detail['sources'])
    print('Zero-overlap answer:', detail['answer'])
    print()
for filename in ['refund_policy.txt', 'payment_policy.txt']:
    print('Boundary evidence for', filename)
    for label, collection, start in [('zero overlap, words 0-39', chunks_0, 0),
                                      ('zero overlap, words 40-79', chunks_0, 40),
                                      ('10 overlap, words 30-69', chunks_10, 30)]:
        chunk = next(item for item in collection if item['source'] == filename and item['start_word'] == start)
        print(label + ': ' + chunk['text'])
    print()

Q1: What delivery time makes a completed order eligible for a full refund?
Expected fact: more than 45 minutes
Retrieved sources: ['refund_policy.txt', 'refund_policy.txt', 'privacy_policy.txt']
Zero-overlap answer: receives a full refund of the order amount.

Q8: How soon is money returned when a payment fails but is deducted?
Expected fact: 48 hours
Retrieved sources: ['payment_policy.txt', 'refund_policy.txt', 'privacy_policy.txt']
Zero-overlap answer: If a payment fails but money is deducted, the amount is returned automatically
 
1000 must be approved by a human support lead before money is returned.

Boundary evidence for refund_policy.txt
zero overlap, words 0-39: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer
zero overlap, words 40-79: receives a full refund of the order amount. If a completed

Q1 breaks because the delivery condition “more than 45 minutes” is in the refund document's first 40 words, while “receives a full refund” begins at word 40. The zero-overlap model selects the latter fragment without the condition, so the expected fact is absent even though the correct document was retrieved. Q8 similarly separates the failed-payment condition from “within 48 hours,” and its timing chunk is not in the retrieved top three. The chunk covering words 30–69 retains each condition and its consequence together; overlap restores both grounded answers.

## Task 5 — Define the five tools and tool map (4 marks)

`TOOLS_SPEC` gives each tool its name, one-sentence description and JSON-style parameter schema. `TOOL_MAP` resolves the model's named call into a Python function. `get_order` uses a parameterised, read-only SQLite query; `search_policy` reuses this same retrieval implementation. Calculation refunds 100% only above 45 minutes, and action tools independently check completed status, eligibility, the exact amount and the INR 1000 escalation rule. Every simulated request is within the seven-day policy window.

`issue_refund` records an automatic action; `escalate_to_human` records a handoff without issuing money. Each returns a `refund_inr` dictionary and appends a timestamped JSON line to the ledger. The run/step id prevents duplicate local actions when the same checkpoint is replayed.

In [9]:
assert len(TOOLS_SPEC) == 5
assert set(TOOL_MAP) == {spec['name'] for spec in TOOLS_SPEC}
assert all(callable(function) for function in TOOL_MAP.values())
for specification in TOOLS_SPEC:
    assert specification['parameters']['type'] == 'object'
    assert set(specification['parameters']['required']) <= set(specification['parameters']['properties'])
print(json.dumps(TOOLS_SPEC, indent=2))
print('Tool map:', ', '.join(TOOL_MAP))

[
  {
    "name": "get_order",
    "description": "Read an order's status, amount and delivery minutes from SQLite.",
    "parameters": {
      "type": "object",
      "properties": {
        "order_id": {
          "type": "string"
        }
      },
      "required": [
        "order_id"
      ],
      "additionalProperties": false
    }
  },
  {
    "name": "search_policy",
    "description": "Search three policy chunks using TF-IDF cosine similarity.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string"
        }
      },
      "required": [
        "query"
      ],
      "additionalProperties": false
    }
  },
  {
    "name": "calculate_refund",
    "description": "Calculate a full refund for delivery over 45 minutes or zero otherwise.",
    "parameters": {
      "type": "object",
      "properties": {
        "order_amount": {
          "type": "number",
          "minimum": 0
        },
        "delivery_minutes": {
  

In [10]:
order_examples = [get_order(order_id) for order_id in ['ORD-100048', 'ORD-100145', 'ORD-100001', 'ORD-100025', 'ORD-999999']]
print('SQLite order observations:')
print(json.dumps(order_examples, indent=2))
assert get_order('ORD-999999') == {'error': 'order not found'}
refund_cases = pd.DataFrame([
    {'order_amount': 100.0, 'delivery_minutes': minutes,
     'refund_inr': calculate_refund(100.0, minutes)['refund_inr']}
    for minutes in [45, 45.01, 60, None]
])
assert calculate_refund(100, 45)['refund_inr'] == 0
assert calculate_refund(100, 45.01)['refund_inr'] == 100
print('\n45-minute boundary and missing-time results:')
print(refund_cases.to_string(index=False))
assert [result['source'] for result in search_policy(query)['results']] == ['refund_policy.txt'] * 3

SQLite order observations:
[
  {
    "order_id": "ORD-100048",
    "status": "completed",
    "order_amount": 128.25,
    "delivery_minutes": 65.0
  },
  {
    "order_id": "ORD-100145",
    "status": "completed",
    "order_amount": 1999.0,
    "delivery_minutes": 54.0
  },
  {
    "order_id": "ORD-100001",
    "status": "completed",
    "order_amount": 109.65,
    "delivery_minutes": 20.0
  },
  {
    "order_id": "ORD-100025",
    "status": "cancelled",
    "order_amount": 45.0,
    "delivery_minutes": null
  },
  {
    "error": "order not found"
  }
]

45-minute boundary and missing-time results:
 order_amount  delivery_minutes  refund_inr
        100.0             45.00         0.0
        100.0             45.01       100.0
        100.0             60.00       100.0
        100.0               NaN         0.0


In [11]:
# These invalid action requests fail before any ledger write.
invalid_requests = [
    ('high-value automatic refund', 'ORD-100145', 1999.0),
    ('cancelled order', 'ORD-100025', 100.0),
    ('unknown order', 'ORD-999999', 100.0),
    ('incorrect amount', 'ORD-100048', 1.0),
]
action_checks = []
ledger_path = ROOT / 'outputs/refund_ledger.jsonl'
ledger_before_checks = ledger_path.read_text() if ledger_path.exists() else ''
for label, order_id, amount in invalid_requests:
    try:
        issue_refund(order_id, amount)
    except ValueError as error:
        action_checks.append({'request': label, 'result': 'REJECTED', 'reason': str(error)})
    else:
        raise AssertionError('An invalid refund action was issued.')
assert (ledger_path.read_text() if ledger_path.exists() else '') == ledger_before_checks
print(pd.DataFrame(action_checks).to_string(index=False))

                    request   result                                                reason
high-value automatic refund REJECTED  Refunds above INR 1000 must be escalated to a human.
            cancelled order REJECTED                Only completed orders can be refunded.
              unknown order REJECTED           Cannot refund an order that does not exist.
           incorrect amount REJECTED Refund must equal the positive eligible order amount.


## Task 6 — Run the ReAct loop (5 marks)

The loop starts with an `AGENT` system message and the customer's user message. A reply with tool calls is appended as an assistant message containing `content: None` plus the id, name and arguments; after execution, a tool message contains that call id, name and a JSON-string observation. The next model call sees the accumulated observations. A reply with no tools ends the run, and its prefix before the colon supplies `REFUND`, `ESCALATE`, `DENIED` or `NOT_FOUND`.

The next cell executes the complete runner rather than loading fabricated outcomes. It verifies all seven scenario statuses and amounts, writes one batch line per scenario to `agent_runs.jsonl`, records three demonstrations separately in `agent_demos.jsonl`, and generates the complete JSON report. Named batch checkpoints are reused on future runs so reruns do not create endless timestamped files.

In [12]:
report = build_report()
scenario_table = pd.DataFrame(report['scenarios'])[
    ['scenario_id', 'status', 'refund_inr', 'expected_refund_inr', 'llm_calls', 'tokens_used']]
print(scenario_table.to_string(index=False))
assert len(scenario_table) == 7
assert scenario_table['scenario_id'].tolist() == ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7']
assert all(abs(row['refund_inr'] - row['expected_refund_inr']) <= 0.01 for row in report['scenarios'])
print('\nAll seven scenario outcomes and refund amounts match their references.')

scenario_id                  status  refund_inr  expected_refund_inr  llm_calls  tokens_used
         S1                  REFUND      128.25               128.25          5         1077
         S2                  REFUND      135.00               135.00          5         1077
         S3                ESCALATE     1999.00              1999.00          5         1064
         S4                  DENIED        0.00                 0.00          4          749
         S5                  DENIED        0.00                 0.00          3          445
         S6               NOT_FOUND        0.00                 0.00          2          144
         S7 FALLBACK_MAX_ITERATIONS        0.00                 0.00          6         3216

All seven scenario outcomes and refund amounts match their references.


In [13]:
batch_runs = [json.loads(line) for line in (ROOT / 'outputs/agent_runs.jsonl').read_text().splitlines()]
assert len(batch_runs) == 7
s1_result = next(result for result in batch_runs if result['scenario_id'] == 'S1')
assert s1_result['llm_calls'] == 5
print('S1 final answer:', s1_result['final_answer'])
for number, step in enumerate(s1_result['steps'], 1):
    print(f"\nTool step {number}: {step['name']}")
    print('Arguments:', json.dumps(step['arguments']))
    print('Observation:', json.dumps(step['observation']))
print('\nThe final fifth model call returns the REFUND answer without requesting a tool.')

S1 final answer: REFUND: INR 128.25 issued.

Tool step 1: get_order
Arguments: {"order_id": "ORD-100048"}
Observation: {"order_id": "ORD-100048", "status": "completed", "order_amount": 128.25, "delivery_minutes": 65.0}

Tool step 2: search_policy
Arguments: {"query": "refund policy late delivery"}
Observation: {"results": [{"source": "refund_policy.txt", "score": 0.2719281507693355, "text": "order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are"}, {"source": "refund_policy.txt", "score": 0.25137413448459545, "text": "Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer"}, {"source": "refund_policy.txt", "score": 0.24540580878362586, "text": "late

In [14]:
s1_checkpoint = load_checkpoint('part_d_batch_s1')
message_trace = []
for index, message in enumerate(s1_checkpoint['messages']):
    message_trace.append({'message': index, 'role': message['role'],
                          'tool_name': message.get('name', ','.join(call['name'] for call in message.get('tool_calls', []))),
                          'tool_call_id': message.get('tool_call_id', ''),
                          'content_type': type(message.get('content')).__name__})
    if message['role'] == 'tool':
        assert isinstance(message['content'], str)
        json.loads(message['content'])
    if message.get('tool_calls'):
        assert message['role'] == 'assistant' and message['content'] is None
        assert all(set(call) == {'id', 'name', 'arguments'} for call in message['tool_calls'])
print(pd.DataFrame(message_trace).to_string(index=False))
print('\nLedger actions for this batch:')
print(pd.DataFrame(report['ledger_actions_this_batch'])[
    ['run_id', 'tool', 'order_id', 'refund_inr', 'operation_id']].to_string(index=False))
assert [row['tool'] for row in report['ledger_actions_this_batch']].count('escalate_to_human') == 1

 message      role        tool_name tool_call_id content_type
       0    system                                        str
       1      user                                        str
       2 assistant        get_order                  NoneType
       3      tool        get_order       call_1          str
       4 assistant    search_policy                  NoneType
       5      tool    search_policy       call_2          str
       6 assistant calculate_refund                  NoneType
       7      tool calculate_refund       call_3          str
       8 assistant     issue_refund                  NoneType
       9      tool     issue_refund       call_4          str
      10 assistant                                        str

Ledger actions for this batch:
                  run_id              tool   order_id  refund_inr                    operation_id
         part_d_batch_s1      issue_refund ORD-100048      128.25          part_d_batch_s1:step:4
         part_d_batch_s2    

## Task 7 — Stop safely (3 marks)

The maximum-iteration limit counts model calls, not tool executions. After every model call, usage is added to the cumulative token total; exceeding the token limit immediately stops the run before its newly requested tool executes. At the top of each iteration, elapsed wall-clock time is measured with `time.monotonic()` and compared using `>=`. Consequently a zero-second limit always stops before the first model call.

Every fallback returns the completed steps and a short human-handoff summary containing scenario, tools, calls and tokens; detailed observations stay in its checkpoint. S7 must complete all six calls under the default 6000-token budget and then use `FALLBACK_MAX_ITERATIONS`.

In [15]:
s7 = next(result for result in batch_runs if result['scenario_id'] == 'S7')
token_demo = report['token_limit_demo']
timeout_demo = report['timeout_demo']
stop_table = pd.DataFrame([
    {'demonstration': 'S7 default limits', 'status': s7['status'],
     'model_calls': s7['llm_calls'], 'tokens_used': s7['tokens_used'], 'completed_tools': len(s7['steps'])},
    {'demonstration': 'S1 token_limit=200', 'status': token_demo['status'],
     'model_calls': token_demo['llm_calls'], 'tokens_used': token_demo['tokens_used'], 'completed_tools': len(token_demo['steps'])},
    {'demonstration': 'S1 time_limit_seconds=0', 'status': timeout_demo['status'],
     'model_calls': timeout_demo['llm_calls'], 'tokens_used': timeout_demo['tokens_used'], 'completed_tools': len(timeout_demo['steps'])},
])
assert s7['status'] == 'FALLBACK_MAX_ITERATIONS' and s7['llm_calls'] == 6 and s7['tokens_used'] <= 6000
assert token_demo['status'] == 'FALLBACK_TOKEN_LIMIT' and token_demo['tokens_used'] > 200
assert token_demo['tools_executed'] == ['get_order', 'search_policy']
assert 'calculate_refund' not in token_demo['tools_executed']
assert timeout_demo['status'] == 'FALLBACK_TIMEOUT' and timeout_demo['llm_calls'] == 0 and timeout_demo['steps'] == []
print(stop_table.to_string(index=False))
print('\nToken stop prevented the newly requested calculate_refund tool.')
for result in [s7, token_demo, timeout_demo]:
    print(result['status'] + ': ' + result['final_answer'])

          demonstration                  status  model_calls  tokens_used  completed_tools
      S7 default limits FALLBACK_MAX_ITERATIONS            6         3216                6
     S1 token_limit=200    FALLBACK_TOKEN_LIMIT            3          462                2
S1 time_limit_seconds=0        FALLBACK_TIMEOUT            0            0                0

Token stop prevented the newly requested calculate_refund tool.
FALLBACK_MAX_ITERATIONS: FALLBACK: Maximum model-call iterations reached without resolution. Human support should review scenario S7; completed search_policy, search_policy, search_policy, search_policy, search_policy, search_policy. Calls=6, tokens=3216. Full observations are retained in the checkpoint.
FALLBACK_TOKEN_LIMIT: FALLBACK: Token budget exceeded after the latest model call. Human support should review scenario S1; completed get_order, search_policy. Calls=3, tokens=462. Full observations are retained in the checkpoint.
FALLBACK_TIMEOUT: FALLBACK: Wall-c

## Task 8 — Checkpoints, crash and resume (3 marks)

Every appended observation is followed by an atomic checkpoint containing run id, scenario id, full messages, cumulative tokens, model-call count, step number and executed-tool list. The simulated crash occurs **after** the second observation has been saved. `resume_agent(run_id)` loads that history and continues with the third tool; it retains calls and tokens from before the crash.

The next cells intentionally crash and resume S1 using the same named demo id. A completed checkpoint returns its saved result on repeat resume, and the ledger's stable run/step key also prevents duplicate local side effects. This demonstrates exactly once execution across the specified saved-checkpoint crash; production concurrent payment processing would need a transactional store and an external provider's idempotency key.

In [16]:
scenario_rows = pd.read_csv(ROOT / 'data/raw/agent_scenarios.csv')
s1_message = scenario_rows.loc[scenario_rows['scenario_id'] == 'S1', 'message'].iloc[0]
crash_run_id = 'part_d_demo_crash_resume'
try:
    run_agent(s1_message, 'S1', run_id=crash_run_id, crash_after_tool_step=2)
except SimulatedCrash as error:
    print('Intentional crash caught:', error)
else:
    raise AssertionError('The crash exercise did not stop after two tools.')
checkpoint_before_resume = load_checkpoint(crash_run_id)
required_fields = {'run_id', 'scenario_id', 'messages', 'tokens_used', 'step_number', 'tools_executed', 'llm_calls'}
assert required_fields <= set(checkpoint_before_resume)
assert checkpoint_before_resume['step_number'] == 2
assert checkpoint_before_resume['tools_executed'] == ['get_order', 'search_policy']
print('Checkpoint fields:', ', '.join(sorted(checkpoint_before_resume)))
print('Saved tools:', checkpoint_before_resume['tools_executed'])
print('Saved model calls:', checkpoint_before_resume['llm_calls'])
print('Saved tokens:', checkpoint_before_resume['tokens_used'])

Intentional crash caught: Simulated crash after saved tool step 2 (search_policy).
Checkpoint fields: elapsed_seconds, limits, llm_calls, messages, pending_tool_calls, run_id, scenario_id, step_number, steps, tokens_used, tools_executed
Saved tools: ['get_order', 'search_policy']
Saved model calls: 2
Saved tokens: 174


In [17]:
resumed_result = resume_agent(crash_run_id)
expected_sequence = ['get_order', 'search_policy', 'calculate_refund', 'issue_refund']
assert resumed_result['tools_executed'] == expected_sequence
assert len(resumed_result['steps']) == 4 and len(set(resumed_result['tools_executed'])) == 4
assert resumed_result['status'] == 'REFUND' and refund_from_result(resumed_result) == 128.25
assert resumed_result['llm_calls'] == 5 and resumed_result['resumed_llm_calls'] == 3
print('Before crash:', ' -> '.join(checkpoint_before_resume['tools_executed']))
print('After resume:', ' -> '.join(resumed_result['tools_executed'][2:]))
print('Complete sequence:', ' -> '.join(resumed_result['tools_executed']))
print('Final:', resumed_result['final_answer'])
print('Cumulative model calls:', resumed_result['llm_calls'])
print('Additional calls after resume:', resumed_result['resumed_llm_calls'])
print('Cumulative estimated tokens:', resumed_result['tokens_used'])

ledger_text_before_repeat = (ROOT / 'outputs/refund_ledger.jsonl').read_text()
assert resume_agent(crash_run_id) == resumed_result
assert (ROOT / 'outputs/refund_ledger.jsonl').read_text() == ledger_text_before_repeat
crash_actions = [json.loads(line) for line in ledger_text_before_repeat.splitlines()
                 if json.loads(line).get('run_id') == crash_run_id]
assert len(crash_actions) == 1 and crash_actions[0]['refund_inr'] == 128.25
print('Ledger actions for this crash/resume run:', len(crash_actions))
print('Repeating resume leaves the ledger unchanged: PASS')

Before crash: get_order -> search_policy
After resume: calculate_refund -> issue_refund
Complete sequence: get_order -> search_policy -> calculate_refund -> issue_refund
Final: REFUND: INR 128.25 issued.
Cumulative model calls: 5
Additional calls after resume: 3
Cumulative estimated tokens: 1077
Ledger actions for this crash/resume run: 1
Repeating resume leaves the ledger unchanged: PASS


## Task 9 — ReAct versus plan-and-execute (2 marks)

The comparison below uses this run's measured S1, S5 and S6 model calls and estimated token totals. It remains within the required 250 words. Planner savings are described as a design expectation; they are not presented as measured results because the supplied mock implements only the ReAct protocol.

In [18]:
discussion = report['react_vs_plan_and_execute']
assert len(discussion.split()) <= 250
print(discussion)
print('\nWord count:', len(discussion.split()))

The measured ReAct workflow uses 5 model calls and 1077 estimated tokens for S1, 3 calls/445 tokens for the cancelled order S5, and 2 calls/144 tokens for NOT_FOUND S6. Each new call resends accumulated history, so its token cost grows with the number and size of observations. A plan-and-execute design could request the full conditional step list once, then execute validated tools locally and replan only on an error; a normal request could use one planning call plus an optional final response call. Its exact token saving is unmeasured because the supplied mock implements ReAct, not a planner. ReAct adapts immediately to NOT_FOUND or a cancelled status and stops before money is issued. A static plan must branch on those observations or replan; blindly following the remaining steps is unsafe. For this fixed refund workflow I would choose a validated conditional plan executed by deterministic Python rules, with human escalation above INR 1000. It reduces repeated model calls while retaini

## Final verification and reproducible outputs

The single repository README reports the scenario table, fallback statuses, grounding comparison, exact recovery sequence and limitations. This notebook provides the executable walkthrough, while Python modules remain reusable from scripts. Reports and ledgers are text/code artifacts; no images, screenshots, slides or remote API calls are produced.

In [19]:
report_on_disk = json.loads((ROOT / 'outputs/part_d_report.json').read_text())
assert all(report_on_disk['verification'].values())
assert len((ROOT / 'outputs/agent_runs.jsonl').read_text().splitlines()) == 7
assert len((ROOT / 'outputs/agent_demos.jsonl').read_text().splitlines()) == 3
for result in batch_runs:
    assert {'scenario_id', 'status', 'final_answer', 'llm_calls', 'tokens_used', 'steps'} <= set(result)
    assert all({'name', 'arguments', 'observation'} <= set(step) for step in result['steps'])
    assert (ROOT / 'state' / (result['run_id'] + '.json')).is_file()
print(pd.DataFrame([{'acceptance_check': name, 'result': 'PASS'}
                    for name in report_on_disk['verification']]).to_string(index=False))
print('\nSaved text artifacts:')
for relative in report_on_disk['artifacts']:
    print(relative)
print('\nPart D completed: all nine tasks and acceptance checks pass.')

               acceptance_check result
                chunk_reference   PASS
  retrieval_grounding_reference   PASS
       zero_overlap_has_failure   PASS
       seven_expected_scenarios   PASS
six_call_s7_with_default_budget   PASS
        independent_stop_limits   PASS
crash_resume_exactly_four_tools   PASS
 completed_resume_is_idempotent   PASS

Saved text artifacts:
outputs/part_d_report.json
outputs/agent_runs.jsonl
outputs/agent_demos.jsonl
outputs/refund_ledger.jsonl
state/part_d_*.json

Part D completed: all nine tasks and acceptance checks pass.
